In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


### Embedding + Positional Encoding


In [ ]:
class Embedder(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embed(x)


class PositionalEncoder(nn.Module):
    def __init__(self, d_model, max_seq_len=200):
        super().__init__()

        pe = torch.zeros(max_seq_len, d_model)
        for pos in range(max_seq_len):
            for i in range(0, d_model, 2):
                pe[pos, i]   = math.sin(pos / (10000 ** ((2 * i)/d_model)))
                pe[pos, i+1] = math.cos(pos / (10000 ** ((2 * (i + 1))/d_model)))

        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]



### Multi-head Attention

In [ ]:
def attention(q, k, v, mask=None, dropout=None):
    d_k = q.size(-1)

    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    scores = F.softmax(scores, dim=-1)

    if dropout:
        scores = dropout(scores)

    return torch.matmul(scores, v)


class MultiHeadAttention(nn.Module):
    def __init__(self, heads, d_model, dropout=0.1):
        super().__init__()

        self.d_model = d_model
        self.d_k = d_model // heads
        self.h = heads

        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):

        bs = q.size(0)

        q = self.q_linear(q).view(bs, -1, self.h, self.d_k).transpose(1,2)
        k = self.k_linear(k).view(bs, -1, self.h, self.d_k).transpose(1,2)
        v = self.v_linear(v).view(bs, -1, self.h, self.d_k).transpose(1,2)

        scores = attention(q, k, v, mask, self.dropout)
        concat = scores.transpose(1,2).contiguous().view(bs, -1, self.d_model)

        return self.out(concat)



In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=2048, dropout=0.1):
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.dropout(F.relu(self.linear_1(x)))
        return self.linear_2(x)


class Norm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.size = d_model

        self.alpha = nn.Parameter(torch.ones(self.size))
        self.bias  = nn.Parameter(torch.zeros(self.size))
        self.eps = eps

    def forward(self, x):
        norm = x.mean(-1, keepdim=True)
        std  = x.std(-1, keepdim=True)
        return self.alpha * (x - norm) / (std + self.eps) + self.bias



class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        self.norm_1 = Norm(d_model)
        self.norm_2 = Norm(d_model)

        self.attention = MultiHeadAttention(heads, d_model, dropout=dropout)
        self.ff = FeedForward(d_model, dropout=dropout)

        self.dropout_1 = nn.Dropout(dropout)
        self.dropout_2 = nn.Dropout(dropout)

    def forward(self, x, mask):
        x2 = self.norm_1(x)
        x = x + self.dropout_1(self.attention(x2, x2, x2, mask))

        x2 = self.norm_2(x)
        x = x + self.dropout_2(self.ff(x2))

        return x



class DecoderLayer(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()

        self.norm_1 = Norm(d_model)
        self.norm_2 = Norm(d_model)
        self.norm_3 = Norm(d_model)

        self.attn_1 = MultiHeadAttention(heads, d_model)
        self.attn_2 = MultiHeadAttention(heads, d_model)

        self.ff = FeedForward(d_model, dropout=dropout)

        self.dropout_1 = nn.Dropout(dropout)
        self.dropout_2 = nn.Dropout(dropout)
        self.dropout_3 = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, tgt_mask):
        x2 = self.norm_1(x)
        x  = x + self.dropout_1(self.attn_1(x2, x2, x2, tgt_mask))

        x2 = self.norm_2(x)
        x  = x + self.dropout_2(self.attn_2(x2, enc_out, enc_out, src_mask))

        x2 = self.norm_3(x)
        x  = x + self.dropout_3(self.ff(x2))

        return x


class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, N, heads, dropout=0.1):
        super().__init__()

        self.N = N
        self.embed = Embedder(vocab_size, d_model)
        self.pe = PositionalEncoder(d_model)

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, heads, dropout) for _ in range(N)
        ])

        self.norm = Norm(d_model)

    def forward(self, src, mask):
        x = self.embed(src)
        x = self.pe(x)

        for i in range(self.N):
            x = self.layers[i](x, mask)

        return self.norm(x)


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, N, heads, dropout=0.1):
        super().__init__()

        self.N = N
        self.embed = Embedder(vocab_size, d_model)
        self.pe = PositionalEncoder(d_model)

        self.layers = nn.ModuleList([
            DecoderLayer(d_model, heads, dropout) for _ in range(N)
        ])

        self.norm = Norm(d_model)

    def forward(self, tgt, enc_out, src_mask, tgt_mask):
        x = self.embed(tgt)
        x = self.pe(x)

        for i in range(self.N):
            x = self.layers[i](x, enc_out, src_mask, tgt_mask)

        return self.norm(x)



In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=512, N=6, heads=8, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(src_vocab, d_model, N, heads, dropout)
        self.decoder = Decoder(tgt_vocab, d_model, N, heads, dropout)
        self.out = nn.Linear(d_model, tgt_vocab)

    def forward(self, src, tgt, src_mask, tgt_mask):
        e = self.encoder(src, src_mask)
        d = self.decoder(tgt, e, src_mask, tgt_mask)
        return self.out(d)


### Load IWSLT15

In [ ]:
import os

def load_iwslt15_text(path):
    train_en = open(os.path.join(path, "train.en"), encoding="utf8").read().splitlines()
    train_vi = open(os.path.join(path, "train.vi"), encoding="utf8").read().splitlines()

    dev_en = open(os.path.join(path, "tst2012.en"), encoding="utf8").read().splitlines()
    dev_vi = open(os.path.join(path, "tst2012.vi"), encoding="utf8").read().splitlines()

    test_en = open(os.path.join(path, "tst2013.en"), encoding="utf8").read().splitlines()
    test_vi = open(os.path.join(path, "tst2013.vi"), encoding="utf8").read().splitlines()

    print("Loaded IWSLT15:")
    print(" - Train:", len(train_en))
    print(" - Dev  :", len(dev_en))
    print(" - Test :", len(test_en))

    return (train_en, train_vi), (dev_en, dev_vi), (test_en, test_vi)


### TOKENIZER WORD-LEVEL

In [ ]:
from collections import Counter

class SimpleTokenizer:
    def __init__(self, vocab_size=30000, min_freq=2, lower=True):
        self.lower = lower
        self.min_freq = min_freq
        self.vocab_size = vocab_size

        self.PAD = "<pad>"
        self.BOS = "<bos>"
        self.EOS = "<eos>"
        self.UNK = "<unk>"

        self.word2id = {}
        self.id2word = {}

    def norm(self, text):
        return text.lower().strip().split()

    def fit(self, texts):
        freq = Counter()
        for t in texts:
            freq.update(self.norm(t))

        vocab_words = [w for w, f in freq.items() if f >= self.min_freq]
        vocab_words = vocab_words[: self.vocab_size]

        vocab = [self.PAD, self.BOS, self.EOS, self.UNK] + vocab_words
        self.word2id = {w: i for i, w in enumerate(vocab)}
        self.id2word = {i: w for w, i in self.word2id.items()}

    def encode(self, text, max_len=100):
        ids = [self.word2id.get(w, self.word2id[self.UNK]) for w in self.norm(text)]
        ids = ids[:max_len]
        return [self.word2id[self.BOS]] + ids + [self.word2id[self.EOS]]

    def decode(self, ids):
        words = []
        for i in ids:
            w = self.id2word.get(int(i), self.UNK)
            if w not in [self.PAD, self.BOS, self.EOS]:
                words.append(w)
        return " ".join(words)

    def vocab_size_(self):
        return len(self.word2id)


### NMT DATASET

In [ ]:
from torch.utils.data import Dataset

class NMTDataset(Dataset):
    def __init__(self, src_texts, tgt_texts, src_tok, tgt_tok, max_len=100):
        self.src = src_texts
        self.tgt = tgt_texts
        self.src_tok = src_tok
        self.tgt_tok = tgt_tok
        self.max_len = max_len

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        src_ids = self.src_tok.encode(self.src[idx], self.max_len)
        tgt_ids = self.tgt_tok.encode(self.tgt[idx], self.max_len)
        return torch.LongTensor(src_ids), torch.LongTensor(tgt_ids)


### COLLATE + MASK

In [ ]:
def collate_batch(batch):
    src, tgt = zip(*batch)
    src = nn.utils.rnn.pad_sequence(src, batch_first=True, padding_value=0)
    tgt = nn.utils.rnn.pad_sequence(tgt, batch_first=True, padding_value=0)
    return src, tgt


def make_src_mask(src):
    return (src != 0).unsqueeze(1).unsqueeze(2)

def make_tgt_mask(tgt):
    T = tgt.size(1)
    pad_mask = (tgt != 0).unsqueeze(1).unsqueeze(2)
    seq_mask = torch.tril(torch.ones((T, T), device=device)).bool()
    return pad_mask & seq_mask

### TRAINING LOOP

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)

        tgt_in = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        src_mask = make_src_mask(src).to(device)
        tgt_mask = make_tgt_mask(tgt_in).to(device)

        pred = model(src, tgt_in, src_mask, tgt_mask)
        pred = pred.reshape(-1, pred.size(-1))
        tgt_out = tgt_out.reshape(-1)

        loss = criterion(pred, tgt_out)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.3 MB/s eta 0:00:00


### VALIDATION (BLEU)


In [ ]:
import sacrebleu

@torch.no_grad()
def evaluate_bleu(model, dataset, src_tok, tgt_tok, max_samples=200, subword="sentencepiece"):
    model.eval()
    hyps = []
    refs = []

    loader = torch.utils.data.DataLoader(
        dataset, batch_size=1, shuffle=False, collate_fn=collate_batch
    )

    for i, (src, tgt) in enumerate(loader):
        if i >= max_samples:
            break

        # ======== SOURCE ========
        src = src.to(device)
        src_mask = make_src_mask(src)

        # ======== GREEDY DECODE ========
        out_ids = greedy_decode(model, src[0], src_mask[0], tgt_tok)

        # hypothesis decode
        hyp = tgt_tok.decode(out_ids)

        # reference decode
        ref_ids = tgt[0].tolist()
        ref = tgt_tok.decode(ref_ids)

        # ======== DETOKENIZE (SentencePiece, BPE, etc.) ========
        if subword == "sentencepiece":
            hyp = hyp.replace("▁", " ").strip()
            ref = ref.replace("▁", " ").strip()
        else:
            # nếu bạn dùng tokenizer không phải SP thì để nguyên
            hyp = hyp.strip()
            ref = ref.strip()

        # ======== REMOVE PAD, BOS, EOS NẾU tokenizer còn giữ ========
        # tùy tokenizer của bạn, nhưng nếu SP/BPE thì BOS/EOS là <s> </s>
        for bad in ["<pad>", "<s>", "</s>"]:
            hyp = hyp.replace(bad, "").strip()
            ref = ref.replace(bad, "").strip()

        hyps.append(hyp)
        refs.append([ref])

    bleu = sacrebleu.corpus_bleu(hyps, refs)
    return bleu.score


### GREEDY DECODE

In [ ]:
@torch.no_grad()
def greedy_decode(model, src_seq, src_mask, tgt_tok, max_len=80):
    model.eval()

    ys = torch.LongTensor([[tgt_tok.word2id[tgt_tok.BOS]]]).to(device)
    src = src_seq.unsqueeze(0).to(device)

    for _ in range(max_len):
        tgt_mask = make_tgt_mask(ys)
        out = model(src, ys, make_src_mask(src), tgt_mask)
        next_word = out[:, -1, :].argmax(-1).item()

        ys = torch.cat([ys, torch.tensor([[next_word]]).to(device)], dim=1)

        if next_word == tgt_tok.word2id[tgt_tok.EOS]:
            break

    return ys[0].cpu().tolist()


### BEAM SEARCH

In [ ]:
@torch.no_grad()
def beam_search(model, src_seq, src_mask, tgt_tok, beam=5, max_len=80):
    model.eval()

    sequences = [[0.0, [tgt_tok.word2id[tgt_tok.BOS]]]]

    src = src_seq.unsqueeze(0).to(device)

    for _ in range(max_len):
        all_candidates = []

        for score, seq in sequences:
            tgt = torch.LongTensor(seq).unsqueeze(0).to(device)
            tgt_mask = make_tgt_mask(tgt)

            out = model(src, tgt, make_src_mask(src), tgt_mask)
            probs = F.log_softmax(out[:, -1, :], dim=-1).squeeze(0)

            topk = torch.topk(probs, beam)

            for i in range(beam):
                w = topk.indices[i].item()
                s = score + topk.values[i].item()
                new_seq = seq + [w]

                if w == tgt_tok.word2id[tgt_tok.EOS]:
                    return new_seq
                else:
                    all_candidates.append([s, new_seq])

        sequences = sorted(all_candidates, key=lambda x: x[0], reverse=True)[:beam]

    return sequences[0][1]


### Evaluate


In [ ]:
import sacrebleu

def evaluate_test_bleu(model, test_src_texts, test_tgt_texts,
                       src_tok, tgt_tok, max_samples=None,
                       bpe_type="sentencepiece"):
    model.eval()
    hyps = []
    refs = []

    if max_samples is None:
        max_samples = len(test_src_texts)

    for i in range(max_samples):

        # ====== SOURCE ======
        src_text = test_src_texts[i]
        tgt_text = test_tgt_texts[i]

        # encode EN
        src_ids = torch.LongTensor(src_tok.encode(src_text)).unsqueeze(0).to(device)
        src_mask = make_src_mask(src_ids)

        # ====== GREEDY DECODE ======
        out_ids = greedy_decode(model, src_ids[0], src_mask[0], tgt_tok)
        hyp = tgt_tok.decode(out_ids)

        # ====== DETOKENIZE ======
        if bpe_type == "sentencepiece":
            hyp = hyp.replace("▁", " ").strip()
            ref = tgt_text.replace("▁", " ").strip()
        else:
            ref = tgt_text.strip()

        hyps.append(hyp)
        refs.append([ref])

    # ====== BLEU ======
    bleu = sacrebleu.corpus_bleu(hyps, refs)
    print(f"TEST BLEU = {bleu.score:.2f}")
    return bleu.score


## train

In [ ]:
def train_pipeline(train_src, train_tgt, val_src, val_tgt,
                   model_name="model", epochs=20, batch_size=32,
                   patience=5):

    # === tokenizer ===
    src_tok = SimpleTokenizer()
    tgt_tok = SimpleTokenizer()
    src_tok.fit(train_src)
    tgt_tok.fit(train_tgt)

    # === datasets ===
    train_ds = NMTDataset(train_src, train_tgt, src_tok, tgt_tok)
    val_ds   = NMTDataset(val_src,   val_tgt,   src_tok, tgt_tok)

    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_batch
    )
    val_loader = torch.utils.data.DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_batch
    )

    # === model ===
    model = Transformer(
        src_tok.vocab_size_(), tgt_tok.vocab_size_(),
        d_model=256, N=4, heads=4
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss(ignore_index=src_tok.word2id[src_tok.PAD])

    best_val_loss = float("inf")
    patience_counter = 0
    best_path = f"{model_name}_best.pt"

    # training loop
    for ep in range(epochs):

        model.train()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)

        model.eval()
        total_val_loss = 0

        with torch.no_grad():
            for src, tgt in val_loader:
                src, tgt = src.to(device), tgt.to(device)
                src_mask = make_src_mask(src)
                tgt_input = tgt[:, :-1]
                tgt_output = tgt[:, 1:]
                tgt_mask = make_tgt_mask(tgt_input)

                logits = model(src, tgt_input, src_mask, tgt_mask)

                vocab_size = logits.shape[-1]
                loss = criterion(
                    logits.reshape(-1, vocab_size),
                    tgt_output.reshape(-1)
                )
                total_val_loss += loss.item()

        avg_val_loss = total_val_loss / len(val_loader)

        print(f"\nEpoch {ep+1}/{epochs}")
        print(f"Train Loss: {train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            torch.save(model.state_dict(), best_path)
            print(f"  Validation loss improved — model saved!")
        else:
            patience_counter += 1
            print(f"⚠️  Loss did not improve. Patience = {patience_counter}/{patience}")

            if patience_counter >= patience:
                print("Early stopping triggered (no loss improvement).")
                break

    print("\nTraining completed.")
    print(f" Best Val Loss: {best_val_loss:.4f}")
    print(f"Model saved at: {best_path}")

    # load best model before returning
    model.load_state_dict(torch.load(best_path))

    return model, src_tok, tgt_tok


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


### Train IWSLT

In [ ]:
path = "/content/drive/MyDrive/Assignment_nlp/data"

(train_en, train_vi), (dev_en, dev_vi), (test_en, test_vi) = load_iwslt15_text(path)


Loaded IWSLT15:
 - Train: 133317
 - Dev  : 1553
 - Test : 1268


In [ ]:
model_iwslt, tok_iwslt_en, tok_iwslt_vi = train_pipeline(
    train_en, train_vi,
    dev_en, dev_vi,
    model_name="iwslt_model"
)


KeyboardInterrupt: 

In [ ]:
src_tok = SimpleTokenizer()
tgt_tok = SimpleTokenizer()

src_tok.fit(train_en)
tgt_tok.fit(train_vi)

print("SRC vocab size:", src_tok.vocab_size_())
print("TGT vocab size:", tgt_tok.vocab_size_())


SRC vocab size: 29345
TGT vocab size: 12517


In [ ]:
evaluate_test_bleu(model_iwslt, test_en, test_vi, tok_iwslt_en, tok_iwslt_vi)

TEST BLEU = 38.91


38.908581870337855

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/Assignment_nlp/iwslt_model_best.pt"   # đúng tên model của bạn

model = Transformer(
    src_tok.vocab_size_(),
    tgt_tok.vocab_size_(),
    d_model=256,
    N=4,
    heads=4
).to(device)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

print("Loaded model:", MODEL_PATH)
@torch.no_grad()
def translate_en_vi(model, sentence, src_tok, tgt_tok, max_len=80):
    model.eval()

    # encode source
    src_ids = torch.LongTensor(
        src_tok.encode(sentence)
    ).unsqueeze(0).to(device)

    src_mask = make_src_mask(src_ids)

    # start with <bos>
    ys = torch.LongTensor(
        [[tgt_tok.word2id[tgt_tok.BOS]]]
    ).to(device)

    for _ in range(max_len):
        tgt_mask = make_tgt_mask(ys)
        out = model(src_ids, ys, src_mask, tgt_mask)

        next_id = out[:, -1].argmax(-1).item()
        ys = torch.cat([ys, torch.LongTensor([[next_id]]).to(device)], dim=1)

        if next_id == tgt_tok.word2id[tgt_tok.EOS]:
            break

    return tgt_tok.decode(ys[0].cpu().tolist())


✅ Loaded model: /content/drive/MyDrive/Assignment_nlp/iwslt_model_best.pt


### Test

In [ ]:
test_sentences = [
    "I am a student at UET and currently live in Hanoi.",
    "In school , we spent a lot of time studying the history of Kim Il-Sung , but we never learned much about the outside world , except that America , South Korea , Japan are the enemies .",
    "Although I often wondered about the outside world , I thought I would spend my entire life in North Korea , until everything suddenly changed .",
    "Deep learning models are powerful ."
]

for s in test_sentences:
    vi = translate_en_vi(model, s, src_tok, tgt_tok)
    print(f"EN: {s}")
    print(f"VI: {vi}")
    print("-"*40)


EN: I am a student at UET and currently live in Hanoi.
VI: tôi là sinh viên ở <unk> và hiện tại đang sống ở <unk>
----------------------------------------
EN: In school , we spent a lot of time studying the history of Kim Il-Sung , but we never learned much about the outside world , except that America , South Korea , Japan are the enemies .
VI: ở trường học , chúng tôi dành rất nhiều thời gian nghiên cứu lịch sử của kim <unk> , nhưng chúng tôi chưa bao giờ biết được nhiều về thế giới bên ngoài , ngoại trừ nước mỹ , nhật bản là người đức .
----------------------------------------
EN: Although I often wondered about the outside world , I thought I would spend my entire life in North Korea , until everything suddenly changed .
VI: mặc dù tôi thường tự hỏi về thế giới bên ngoài , tôi nghĩ rằng tôi sẽ dành toàn bộ cuộc đời mình ở bắc triều tiên , cho đến khi mọi thứ thay đổi .
----------------------------------------
EN: Deep learning models are powerful .
VI: các mô hình sâu sắc là quyền 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sacrebleu
import torch

@torch.no_grad()
def evaluate_test_bleu_standard(
    model,
    test_src_texts,
    test_tgt_texts_RAW,
    src_tok,
    tgt_tok,
    max_len=80
):
    model.eval()
    hyps = []
    refs = []

    for i in range(len(test_src_texts)):
        src_text = test_src_texts[i]

        src_ids = torch.LongTensor(
            src_tok.encode(src_text)
        ).unsqueeze(0).to(device)

        src_mask = make_src_mask(src_ids)

        out_ids = greedy_decode(
            model,
            src_ids[0],
            src_mask[0],
            tgt_tok,
            max_len=max_len
        )

        hyp = tgt_tok.decode(out_ids).strip()

        ref = test_tgt_texts_RAW[i].strip()

        hyps.append(hyp)
        refs.append([ref])

    bleu = sacrebleu.corpus_bleu(
        hyps,
        refs,
        tokenize="13a"
    )

    print(f"STANDARD BLEU = {bleu.score:.2f}")
    return bleu.score


In [ ]:
(train_en, train_vi), (dev_en, dev_vi), (test_en, test_vi) = load_iwslt15_text(path)

src_tok = SimpleTokenizer()
tgt_tok = SimpleTokenizer()

src_tok.fit(train_en)
tgt_tok.fit(train_vi)


Loaded IWSLT15:
 - Train: 133317
 - Dev  : 1553
 - Test : 1268


In [ ]:
Transformer(
    src_tok.vocab_size_(),
    tgt_tok.vocab_size_(),
    d_model=256,
    N=4,
    heads=4
)


Transformer(
  (encoder): Encoder(
    (embed): Embedder(
      (embed): Embedding(29345, 256)
    )
    (pe): PositionalEncoder()
    (layers): ModuleList(
      (0-3): 4 x EncoderLayer(
        (norm_1): Norm()
        (norm_2): Norm()
        (attention): MultiHeadAttention(
          (q_linear): Linear(in_features=256, out_features=256, bias=True)
          (k_linear): Linear(in_features=256, out_features=256, bias=True)
          (v_linear): Linear(in_features=256, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (out): Linear(in_features=256, out_features=256, bias=True)
        )
        (ff): FeedForward(
          (linear_1): Linear(in_features=256, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear_2): Linear(in_features=2048, out_features=256, bias=True)
        )
        (dropout_1): Dropout(p=0.1, inplace=False)
        (dropout_2): Dropout(p=0.1, inplace=False)
      )
    )
    (norm

In [ ]:
model = Transformer(
    src_tok.vocab_size_(),
    tgt_tok.vocab_size_(),
    d_model=256,
    N=4,
    heads=4
).to(device)

model.load_state_dict(
    torch.load("/content/drive/MyDrive/Assignment_nlp/iwslt_model_best.pt", map_location=device)
)

model.eval()
print("✅ Model loaded successfully")


✅ Model loaded successfully


### BLEU

In [ ]:
evaluate_test_bleu_standard(
    model,
    test_en,
    test_vi,
    src_tok,
    tgt_tok
)


STANDARD BLEU = 38.91


38.908581870337855

### PPL

In [ ]:
import math
import torch
import torch.nn as nn

@torch.no_grad()
def evaluate_test_ppl(
    model,
    test_src_texts,
    test_tgt_texts,
    src_tok,
    tgt_tok,
    batch_size=32
):
    model.eval()

    test_ds = NMTDataset(
        test_src_texts,
        test_tgt_texts,
        src_tok,
        tgt_tok
    )

    test_loader = torch.utils.data.DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_batch
    )

    criterion = nn.CrossEntropyLoss(
        ignore_index=tgt_tok.word2id[tgt_tok.PAD],
        reduction="sum"
    )

    total_loss = 0.0
    total_tokens = 0

    for src, tgt in test_loader:
        src = src.to(device)
        tgt = tgt.to(device)

        tgt_in = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        src_mask = make_src_mask(src)
        tgt_mask = make_tgt_mask(tgt_in)

        logits = model(src, tgt_in, src_mask, tgt_mask)
        vocab_size = logits.size(-1)

        loss = criterion(
            logits.reshape(-1, vocab_size),
            tgt_out.reshape(-1)
        )

        total_loss += loss.item()
        total_tokens += (tgt_out != tgt_tok.word2id[tgt_tok.PAD]).sum().item()

    avg_nll = total_loss / total_tokens
    ppl = math.exp(avg_nll)

    print(f"TEST NLL  = {avg_nll:.4f}")
    print(f"TEST PPL  = {ppl:.2f}")

    return ppl


In [ ]:
path = "/content/drive/MyDrive/Assignment_nlp/data"

(train_en, train_vi), (dev_en, dev_vi), (test_en, test_vi) = load_iwslt15_text(path)
tok_iwslt_en = SimpleTokenizer()
tok_iwslt_vi = SimpleTokenizer()

tok_iwslt_en.fit(train_en)
tok_iwslt_vi.fit(train_vi)


Loaded IWSLT15:
 - Train: 133317
 - Dev  : 1553
 - Test : 1268


In [ ]:
model = Transformer(
    tok_iwslt_en.vocab_size_(),
    tok_iwslt_vi.vocab_size_(),
    d_model=256,
    N=4,
    heads=4
).to(device)

model.load_state_dict(
    torch.load("/content/drive/MyDrive/Assignment_nlp/iwslt_model_best.pt", map_location=device)
)

model.eval()
print(" Model loaded")


✅ Model loaded


In [ ]:
evaluate_test_ppl(
    model,
    test_en,
    test_vi,
    tok_iwslt_en,
    tok_iwslt_vi
)


TEST NLL  = 2.2995
TEST PPL  = 9.97


9.96905127596569